## 0. Install dependencies (added -- Colab's default runtime doesn't include these)


In [ ]:
# ADDED: sb3_contrib (RecurrentPPO) and gymnasium are not part of Colab's default
# image -- the original notebook assumed a persistent local environment where these
# had already been pip-installed once. A fresh/restarted Colab runtime needs this
# every time, since the runtime's installed packages don't persist across restarts
# (only files on Drive do). sb3-contrib pulls in stable-baselines3 automatically.
!pip install -q sb3-contrib gymnasium
print('Dependencies installed.')


# NB4 -- Attack 2: PPO/RL Black-Box Evasion Agent (Multiclass, v4)

Rebuilt against the v4 corrected pipeline from the original leakage-era
`NB4_Attack2_Drive_results.ipynb` architectural reference. **Only the data pipeline and
target model source are corrected; the attack architecture (Gym environment, reward
design, RecurrentPPO config, evaluation loop) is carried over unchanged.**

**Corrections applied vs. the original reference notebook:**
- Data: loads `phase1_data_v4.pkl` (temporal+embargo split, 43 features) instead of the
  inline random 80/20 stratified split on 25 features.
- Target model: retrains the Optuna-tuned XGBoost from the actual v4 `best_params`
  (multiclass uses tuned, matching NB3's model-selection asymmetry).
- `CYBER_IDX` is derived by name lookup (`FEATURE_COLS.index(...)`), not position -- this
  means the 18 new v4 temporal features (none of which are cyber-named) are
  automatically excluded from the action space with no code changes needed.
- Attack 1 comparison cell loads real PGD numbers from `attack1_results_v4.pkl` instead
  of hardcoded stale values.
- Paths: local Windows, not Colab/Drive.
- **Dropped**: the original notebook's cell 20, a second `ppo_results` dict with
  suspiciously round numbers that didn't match the actual evaluation loop's output and
  was never referenced again -- looked like leftover dead code, not carried forward.

**Unchanged from the reference architecture:**
- `IDSEvasionEnv`: black-box Gym environment, action space restricted to the 10
  cyber-named features, reward = evasion bonus minus perturbation-magnitude penalty,
  10-step episode horizon.
- `RecurrentPPO` (MlpLstmPolicy, LSTM hidden=64, 1 layer), 50,000 training timesteps,
  trained on attack-class-only samples.
- Full attack-class test-set evaluation loop, per-class evasion rate + mean
  steps-to-evade.

---

**This revision (Attack 2, baseline-model variant):** the original NB4 attacks the Optuna-tuned XGBoost (test F1=0.7110), matching Attack 1's original target model. This revision retargets the identical PPO methodology (Gym environment, reward design, RecurrentPPO config, unconstrained + budget-matched evaluation, 5-seed robustness) at the untuned baseline model (test F1=0.7225, Table 6.1's recommended headline model) instead, using the exact hyperparameters from `2_NB1_Baseline_MultiClass_v4`. Results save to separate `*_baseline_v4.*` filenames so they don't overwrite the original tuned-model run. The Attack 1 baseline check already found a large, class-specific discrepancy for backdoor under PGD (49.23% tuned vs 30.17% untuned) while confirming ransomware/reverseshell physics-immunity holds under both models; this notebook checks whether that same pattern -- immunity robust, but some classes' exact evasion numbers model-dependent -- also holds for the PPO attack, both unconstrained and budget-matched. A comparison cell near the end contrasts both runs directly if both output files are present.

**Note on runtime:** this notebook trains multiple RecurrentPPO agents from scratch (unconstrained, budget-matched, and 5 more for the seed-robustness check -- 7 full PPO training runs total, same as the original NB4). Expect this to take as long as the original NB4 run did; nothing about the baseline model changes the RL training cost.


---

**Further fixed (this revision) -- disconnect resilience:** a prior run disconnected partway through and had to restart from scratch, because (a) trained PPO agents were never saved to disk -- they only existed in kernel memory -- and (b) the per-sample evaluation loops (81,718 samples, up to 10 sequential model calls each, in pure Python) ran silently with no progress output and no intermediate saves, so there was no way to tell how far a run had gotten before it died. Fixed by: (1) saving every trained PPO agent (`ppo_evasion_agent_unconstrained_baseline_v4`, `ppo_evasion_agent_budgeted_baseline_v4`, plus a `.load()` reload check) immediately after training, before the slow evaluation starts; (2) checkpointing evaluation progress every 2,000 samples with a resume-from-checkpoint on re-run, for both the unconstrained and budget-matched evaluation loops; (3) the 5-seed robustness cell (the most expensive cell in the notebook -- 5 full train+evaluate cycles) now checkpoints per seed, so a disconnect after seed 4 of 5 does not require redoing seeds 1-3; (4) periodic progress printing (samples/sec, estimated time remaining) so a long-running cell is visibly alive rather than silent. If this notebook disconnects now, simply re-run the cell it stopped on (or re-run the whole notebook top to bottom) -- completed work will be detected and skipped rather than redone.


---

**Further fixed (this revision) -- mid-training resumability:** a run disconnected *during* training itself (confirmed: died at 19,122/50,000 timesteps), which the previous revision's fix didn't cover -- it only saved the model after `learn()` returned, so a mid-training disconnect still lost the entire partial run. Fixed using SB3's `CheckpointCallback`: cells 5, 8, and each seed's training in cell 12 now save a checkpoint every 5,000 timesteps *during* training, and on re-run, automatically detect the latest checkpoint and resume from there (`RecurrentPPO.load(..., env=env)` followed by `learn(total_timesteps=remaining, reset_num_timesteps=False)`) instead of retraining from timestep 0. Checkpoints are cleaned up automatically once each training run completes and the final model is safely saved under its real filename. Net effect: a disconnect anywhere in this notebook now costs at most ~5,000 timesteps of lost progress (about a minute or two), not an entire training run.


## Setup

## Environment setup -- run once, in a terminal, BEFORE opening this notebook

**Do not run pip installs from inside a Jupyter cell in this environment** -- installing `stable-baselines3`/`sb3-contrib` can trigger a dependency-resolution torch reinstall, and Windows cannot overwrite torch's DLLs while they're loaded in a running kernel. This previously broke the environment (`ImportError: DLL load failed while importing _C`).

In a fresh Anaconda Prompt, with no Jupyter kernel running against this env:

```
conda activate gnn_ot_ids
pip install stable-baselines3 sb3-contrib gymnasium --no-deps
```

`--no-deps` stops pip from trying to touch the existing torch install. If an import error mentions a missing small package (e.g. `cloudpickle`), install it individually -- do not let it reinstall torch.

Once that's done, restart the kernel and run this notebook normally from the cell below.

In [1]:
import os, random
import numpy as np
import pandas as pd
import joblib
import torch
import xgboost as xgb
from sklearn.metrics import f1_score
import gymnasium as gym
from gymnasium import spaces
from sb3_contrib import RecurrentPPO
import warnings
warnings.filterwarnings('ignore')

# CHANGED: PHYSAR-NEW now lives on Google Drive, not a local Windows path --
# switching to this matters beyond convenience: if Colab does a full runtime
# restart (not just a stalled cell), the local VM disk is wiped completely,
# which would also wipe the mid-training checkpoints added earlier in this
# notebook. Mounting Drive means checkpoints, saved models, and results all
# survive a runtime restart, not just a network hiccup.
from google.colab import drive
drive.mount('/content/drive')

# Adjust this if PHYSAR-NEW sits in a different location in your Drive (e.g. inside
# a subfolder) -- check the left-hand file browser in Colab to confirm the exact path.
PROJECT_DIR = '/content/drive/MyDrive/PHYSAR-NEW'
os.chdir(PROJECT_DIR)

# Fail fast with a clear message if the path is wrong, rather than a confusing
# FileNotFoundError several cells later when phase1_data_v4.pkl fails to load.
_expected_file = f'phase1_data_v4.pkl'
if not os.path.exists(_expected_file):
    print(f'WARNING: {_expected_file} not found in {PROJECT_DIR}')
    print('Contents of this directory:')
    for _f in sorted(os.listdir(PROJECT_DIR))[:20]:
        print(f'  {_f}')
    print('Update PROJECT_DIR above to point at the correct PHYSAR-NEW folder in your Drive.')
else:
    print(f'Found {_expected_file} -- PROJECT_DIR looks correct.')

RANDOM_STATE = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'Working directory: {os.getcwd()}')
print(f'Seeds set to {RANDOM_STATE}')

# GPU detection for XGBoost -- same pattern as NB2/NB3
import warnings as _warnings
with _warnings.catch_warnings(record=True) as _caught:
    _warnings.simplefilter('always')
    _test_model = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2)
    _test_model.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _fell_back_to_cpu = any('No visible GPU' in str(w.message) for w in _caught)

if _fell_back_to_cpu:
    GPU_PARAMS = {}
    print('GPU not available for XGBoost (silently fell back to CPU) -- using CPU.')
else:
    GPU_PARAMS = {'tree_method': 'hist', 'device': 'cuda'}
    print('GPU detected -- XGBoost will use CUDA (tree_method=hist, device=cuda).')

Working directory: C:\Users\user\Desktop\Projects\PHYSAR-NEW
Seeds set to 42
GPU detected -- XGBoost will use CUDA (tree_method=hist, device=cuda).


## 1. Load Phase 1 data and the tuned XGBoost hyperparameters

Same pattern as NB3 -- loads `phase1_data_v4.pkl` and `optuna_xgboost_results_v4.pkl`'s
saved `best_params`, since no fitted tuned-booster artifact exists on disk for the
multiclass target.

In [2]:
PIPELINE_VERSION = 'v4'
EXPECTED_TEST_FRAC = 0.15
EXPECTED_EMBARGO_SECONDS = 1.0

data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')

version_mismatch = (
    data.get('PIPELINE_VERSION') != PIPELINE_VERSION
    or data.get('TEST_FRAC') != EXPECTED_TEST_FRAC
    or data.get('EMBARGO_SECONDS') != EXPECTED_EMBARGO_SECONDS
)
if version_mismatch:
    print(f"WARNING: loaded file's version metadata doesn't match this notebook's "
          f"expectations -- proceeding anyway, but double check before trusting results.")
    print(f"  expected: version={PIPELINE_VERSION} test_frac={EXPECTED_TEST_FRAC} "
          f"embargo={EXPECTED_EMBARGO_SECONDS}")
    print(f"  loaded:   version={data.get('PIPELINE_VERSION')} test_frac={data.get('TEST_FRAC')} "
          f"embargo={data.get('EMBARGO_SECONDS')}")
else:
    print(f'Version check passed: {PIPELINE_VERSION}, TEST_FRAC={EXPECTED_TEST_FRAC}, '
          f'EMBARGO_SECONDS={EXPECTED_EMBARGO_SECONDS}')

X_train_scaled = data['X_train_scaled']
X_test_scaled  = data['X_test_scaled']
y_train = pd.Series(data['y_train'])
y_test  = pd.Series(data['y_test'])
FEATURE_COLS = data['FEATURE_COLS']

print(f'Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}')
print(f'Features: {len(FEATURE_COLS)}')

# CHANGED FROM ORIGINAL NB4: targets the UNTUNED baseline XGBoost (Table 6.1's
# headline architecture-comparison model, F1=0.7225) rather than the Optuna-tuned
# configuration (F1=0.7110) the original NB4 loads via optuna_xgboost_results_v4.pkl.
# Companion to the Attack 1 baseline check -- same purpose: confirm whether the
# tuned-vs-baseline model choice changes any qualitative PPO finding.
baseline_results = joblib.load(f'baseline_results_{PIPELINE_VERSION}.pkl')
print(f"\nBaseline (untuned) XGBoost reference figures from baseline_results_{PIPELINE_VERSION}.pkl:")
print(f"  Train F1: {baseline_results['xgb_train_f1']:.4f}  Test F1: {baseline_results['xgb_test_f1']:.4f}")


Version check passed: v4, TEST_FRAC=0.15, EMBARGO_SECONDS=1.0
Train: (503260, 43), Test: (97463, 43)
Features: 43

Baseline (untuned) XGBoost reference figures from baseline_results_v4.pkl:
  Train F1: 0.9980  Test F1: 0.7225


## 2. Target/defender model: retrain the tuned XGBoost (same recipe as NB3)

In [3]:
# Baseline (untuned) hyperparameters, copied exactly from 2_NB1_Baseline_MultiClass_v4
# so this reproduces the same Table 6.1 baseline model, not a re-tuned one.
baseline_params = dict(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=6,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **GPU_PARAMS,
)

xgb_model = xgb.XGBClassifier(**baseline_params)
xgb_model.fit(X_train_scaled, y_train, verbose=False)

clean_test_f1 = f1_score(y_test, xgb_model.predict(X_test_scaled), average='weighted')
print(f'XGBoost (untuned baseline, retrained) -- Test F1: {clean_test_f1:.4f}')

recorded_test_f1 = baseline_results['xgb_test_f1']
f1_diff = abs(clean_test_f1 - recorded_test_f1)
if f1_diff > 0.005:
    print(f"\nWARNING: retrained test F1 differs from recorded baseline value by {f1_diff:.4f} -- "
          f"double-check GPU_PARAMS / library versions before trusting this as the target model.")
else:
    print(f'Sanity check passed: retrained test F1 matches recorded baseline test F1 (diff={f1_diff:.4f}).')

# Identify cyber feature indices (attacker-controllable action space) -- by name, not
# position, so this scales correctly to v4's 43 features with no changes needed (the 18
# new temporal features are physics-derived, not cyber-named, so they are automatically
# excluded from the action space).
CYBER_FEATURE_NAMES = [
    'cyber_pkt_count', 'cyber_pktlen_mean', 'cyber_pktlen_std',
    'cyber_n_unique_srcport', 'cyber_n_unique_dstport',
    'cyber_modbus_frac', 'cyber_tls_frac', 'cyber_tcp_frac',
    'cyber_iat_mean', 'cyber_iat_std',
]
CYBER_IDX = [FEATURE_COLS.index(c) for c in CYBER_FEATURE_NAMES]
N_FEATURES = len(FEATURE_COLS)
N_CYBER = len(CYBER_IDX)
print(f'Total features: {N_FEATURES}, attacker-controllable (cyber) features: {N_CYBER}')
print(f'Cyber feature indices: {CYBER_IDX}')


XGBoost (untuned baseline, retrained) -- Test F1: 0.7256
Sanity check passed: retrained test F1 matches recorded baseline test F1 (diff=0.0031).
Total features: 43, attacker-controllable (cyber) features: 10
Cyber feature indices: [11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


## 3. Custom Gym environment: black-box evasion against XGBoost

Unchanged from the reference architecture. Action space restricted to the 10 cyber
features; observation is current features + last perturbation + last-prediction
one-hot (6 classes) + normalised step count.

In [4]:
EPS_BOUND = 0.3       # per-step perturbation bound (standardised units)
MAX_STEPS = 10
EVASION_REWARD = 10.0
STEP_PENALTY_SCALE = 0.01
TIMEOUT_PENALTY = -1.0


class IDSEvasionEnv(gym.Env):
    """Black-box evasion environment against the tuned XGBoost IDS."""

    metadata = {'render_modes': []}

    def __init__(self, X_pool, y_pool, model, cyber_idx, n_features, n_classes=6,
                 eps_bound=EPS_BOUND, max_steps=MAX_STEPS, seed=None,
                 eps_total_budget=None):
        super().__init__()
        self.X_pool = X_pool          # attack-class samples only
        self.y_pool = y_pool
        self.model = model
        self.cyber_idx = np.array(cyber_idx)
        self.n_features = n_features
        self.n_cyber = len(cyber_idx)
        self.n_classes = n_classes
        self.eps_bound = eps_bound
        self.max_steps = max_steps
        self.rng = np.random.RandomState(seed)
        # Optional CUMULATIVE perturbation budget, matching PGD's L_inf ball around
        # x_orig. Default None preserves the original (unconstrained-across-steps)
        # behaviour; set to e.g. 0.1 for a budget-matched comparison against Attack 1's
        # PGD, whose perturbation is clamped to +/-eps at every step (see pgd_attack in
        # NB3). Without this, PPO's per-step bound (eps_bound=0.3) already exceeds
        # PGD's entire budget in a single step, and can reach ~max_steps*eps_bound
        # cumulative drift -- not an apples-to-apples attack-budget comparison.
        self.eps_total_budget = eps_total_budget

        self.action_space = spaces.Box(
            low=-eps_bound, high=eps_bound, shape=(self.n_cyber,), dtype=np.float32
        )

        obs_dim = self.n_features + self.n_cyber + self.n_classes + 1
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )

    def _get_obs(self):
        last_pred_onehot = np.zeros(self.n_classes, dtype=np.float32)
        last_pred_onehot[self.last_pred] = 1.0
        step_norm = np.array([self.step_count / self.max_steps], dtype=np.float32)
        return np.concatenate([
            self.x_current.astype(np.float32),
            self.last_perturbation.astype(np.float32),
            last_pred_onehot,
            step_norm,
        ])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.rng.randint(0, len(self.X_pool))
        self.x_orig = self.X_pool[idx].copy()
        self.x_current = self.x_orig.copy()
        self.y_true = self.y_pool[idx]
        self.step_count = 0
        self.last_perturbation = np.zeros(self.n_cyber, dtype=np.float32)

        pred = self.model.predict(self.x_current.reshape(1, -1))[0]
        self.last_pred = pred

        return self._get_obs(), {}

    def step(self, action):
        action = np.clip(action, -self.eps_bound, self.eps_bound)

        proposed = self.x_current[self.cyber_idx] + action
        if self.eps_total_budget is not None:
            # Clamp CUMULATIVE displacement from x_orig to the total budget, same
            # ball PGD is constrained to -- the agent can still use up to max_steps
            # to search within this budget, it just can't exceed it in total.
            cumulative = proposed - self.x_orig[self.cyber_idx]
            cumulative = np.clip(cumulative, -self.eps_total_budget, self.eps_total_budget)
            proposed = self.x_orig[self.cyber_idx] + cumulative
            action = proposed - self.x_current[self.cyber_idx]  # actual applied delta, for reward/obs

        self.x_current[self.cyber_idx] = proposed
        self.last_perturbation = action.astype(np.float32)
        self.step_count += 1

        pred = self.model.predict(self.x_current.reshape(1, -1))[0]
        self.last_pred = pred

        terminated = False
        truncated = False

        if pred == 0:  # evasion success: predicted benign
            reward = EVASION_REWARD - STEP_PENALTY_SCALE * np.sum(action ** 2)
            terminated = True
        elif self.step_count >= self.max_steps:
            reward = TIMEOUT_PENALTY - STEP_PENALTY_SCALE * np.sum(action ** 2)
            truncated = True
        else:
            reward = -STEP_PENALTY_SCALE * np.sum(action ** 2)

        return self._get_obs(), reward, terminated, truncated, {'evaded': pred == 0}

## 4. Training pool: attack-class samples from the training set

In [5]:
attack_mask_train = (y_train.values != 0)
X_attack_train = X_train_scaled[attack_mask_train]
y_attack_train = y_train.values[attack_mask_train]

print(f'Training pool (attack-class samples): {X_attack_train.shape}')
print(f'Class distribution: {np.bincount(y_attack_train)}')

Training pool (attack-class samples): (419101, 43)
Class distribution: [    0 85731 79901 84345 84430 84694]


## 5. Train PPO agent (RecurrentPPO with LSTM policy)

In [ ]:
from stable_baselines3.common.callbacks import CheckpointCallback
import hashlib

# ADDED: fingerprinting, so a checkpoint from a DIFFERENT model/config never gets
# silently resumed as if it were consistent with the current run. Without this,
# switching which XGBoost model is attacked, changing a hyperparameter, or re-running
# an old notebook version and finding leftover checkpoint files would all resume
# training on top of stale, inconsistent progress with no warning.
def _fingerprint_config(*parts):
    """Hash a tuple of config values into a short hex digest."""
    return hashlib.sha256(repr(parts).encode()).hexdigest()[:16]

def _fingerprint_file(path):
    """Hash a file's bytes into a short hex digest."""
    with open(path, 'rb') as _f:
        return hashlib.sha256(_f.read()).hexdigest()[:16]

def _model_fingerprint(model, X_probe):
    """Fingerprint the ACTUAL model being attacked by hashing its predictions on a
    fixed probe set -- catches baseline-vs-tuned swaps or any other change to
    xgb_model, regardless of how the model object itself was constructed."""
    preds = model.predict(X_probe)
    return hashlib.sha256(np.asarray(preds).tobytes()).hexdigest()[:16]

# ADDED: mid-training checkpointing. The previous fix only saved the model AFTER
# learn() finished -- if the connection drops DURING training (confirmed: a run
# died at 19,122/50,000 timesteps), that entire partial run was still lost. This
# saves a checkpoint every CKPT_FREQ timesteps during training itself, and resumes
# from the latest one on re-run instead of starting the 50,000 steps over.

TOTAL_TIMESTEPS = 50000
CKPT_FREQ = 5000
CKPT_DIR = f'_ppo_train_checkpoints_unconstrained_{PIPELINE_VERSION}'
os.makedirs(CKPT_DIR, exist_ok=True)

env = IDSEvasionEnv(
    X_pool=X_attack_train, y_pool=y_attack_train,
    model=xgb_model, cyber_idx=CYBER_IDX, n_features=N_FEATURES, n_classes=6,
    seed=RANDOM_STATE,
)

def _latest_checkpoint(ckpt_dir):
    """Find the highest-timestep checkpoint saved by CheckpointCallback, if any.
    CheckpointCallback names files like '{prefix}_{num_timesteps}_steps.zip'."""
    if not os.path.isdir(ckpt_dir):
        return None
    candidates = [f for f in os.listdir(ckpt_dir) if f.endswith('_steps.zip')]
    if not candidates:
        return None
    def _steps(fname):
        return int(fname.rsplit('_steps.zip', 1)[0].rsplit('_', 1)[1])
    best = max(candidates, key=_steps)
    return os.path.join(ckpt_dir, best), _steps(best)

_current_fp = _fingerprint_config(
    _model_fingerprint(xgb_model, X_attack_train[:200]),
    EPS_BOUND, MAX_STEPS, EVASION_REWARD, STEP_PENALTY_SCALE, TIMEOUT_PENALTY, RANDOM_STATE,
)
_fp_path = os.path.join(CKPT_DIR, '_fingerprint.txt')

_found = _latest_checkpoint(CKPT_DIR)
if _found is not None and os.path.exists(_fp_path) and open(_fp_path).read().strip() == _current_fp:
    _ckpt_path, _ckpt_steps = _found
    print(f'Found mid-training checkpoint at {_ckpt_steps}/{TOTAL_TIMESTEPS} timesteps '
          f'(fingerprint matches current config) -- resuming.')
    ppo_model = RecurrentPPO.load(_ckpt_path, env=env)
    _remaining = TOTAL_TIMESTEPS - ppo_model.num_timesteps
else:
    if _found is not None:
        print('Found a checkpoint, but its fingerprint does NOT match the current model/config '
              '-- treating as stale and discarding it rather than risk resuming inconsistent '
              'progress.')
        import shutil as _shutil_stale
        _shutil_stale.rmtree(CKPT_DIR)
        os.makedirs(CKPT_DIR, exist_ok=True)
    print(f'No checkpoint found -- starting fresh, 0/{TOTAL_TIMESTEPS} timesteps.')
    with open(_fp_path, 'w') as _f:
        _f.write(_current_fp)
    ppo_model = RecurrentPPO(
        'MlpLstmPolicy',
        env,
        n_steps=128,
        batch_size=64,
        learning_rate=3e-4,
        gamma=0.99,
        seed=RANDOM_STATE,
        verbose=0,
        policy_kwargs=dict(lstm_hidden_size=64, n_lstm_layers=1),
    )
    _remaining = TOTAL_TIMESTEPS

if _remaining > 0:
    ckpt_callback = CheckpointCallback(
        save_freq=CKPT_FREQ, save_path=CKPT_DIR, name_prefix='ppo_unconstrained',
    )
    ppo_model.learn(total_timesteps=_remaining, reset_num_timesteps=False,
                     callback=ckpt_callback, progress_bar=True)
else:
    print('Training already complete per checkpoint -- skipping learn().')

# Save the final model under its real name, then clean up the intermediate
# per-step checkpoints -- keeping them around risks a future run resuming from
# stale progress after the real final model has already changed.
ppo_model.save('ppo_evasion_agent_unconstrained_baseline_v4')
print("Saved: ppo_evasion_agent_unconstrained_baseline_v4.zip")

_check = RecurrentPPO.load('ppo_evasion_agent_unconstrained_baseline_v4')
print('Reload check passed -- model file is valid and loadable.')
del _check

import shutil as _shutil
if os.path.exists(CKPT_DIR):
    _shutil.rmtree(CKPT_DIR)
    print(f'Removed training checkpoints: {CKPT_DIR} (superseded by final saved model)')

print('PPO training complete.')


Output()

## 6. Evaluation: per-class evasion rate on held-out test set

In [ ]:
# ADDED: checkpointing + resume, plus periodic progress printing. Previously this
# loop ran all 81,718 samples with zero output and zero saved state until the very
# end -- a disconnect anywhere in the loop meant starting over from sample 0 with
# no way to tell how far it had gotten. Now it saves partial progress every
# CHECKPOINT_EVERY samples and resumes automatically if a checkpoint is found.

CHECKPOINT_EVERY = 2000
CHECKPOINT_PATH = f'_checkpoint_unconstrained_eval_{PIPELINE_VERSION}.pkl'

attack_mask_test = (y_test.values != 0)
X_attack_test = X_test_scaled[attack_mask_test]
y_attack_test = y_test.values[attack_mask_test]

eval_env = IDSEvasionEnv(
    X_pool=X_attack_test, y_pool=y_attack_test,
    model=xgb_model, cyber_idx=CYBER_IDX, n_features=N_FEATURES, n_classes=6,
    seed=RANDOM_STATE + 1,
)

class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
N_EVAL_EPISODES = len(X_attack_test)  # evaluate on full attack-class test set

# ADDED: fingerprint the trained model + eval data before trusting a checkpoint.
# Without this, if cell 6 gets re-run and produces a NEW trained model, a leftover
# evaluation checkpoint from the OLD model would resume as if nothing changed --
# silently mixing evaluations from two different models into one result set.
_eval_fp = _fingerprint_config(
    _fingerprint_file('ppo_evasion_agent_unconstrained_baseline_v4.zip'),
    X_attack_test.shape, float(X_attack_test[:10].sum()),
)

if os.path.exists(CHECKPOINT_PATH):
    _ckpt = joblib.load(CHECKPOINT_PATH)
    if _ckpt.get('fingerprint') == _eval_fp:
        evasion_results = _ckpt['evasion_results']
        X_adv_ppo_full = _ckpt['X_adv_ppo_full']
        start_i = _ckpt['next_i']
        print(f'Resuming from checkpoint: {start_i}/{N_EVAL_EPISODES} samples already done '
              f'(fingerprint matches current trained model).')
    else:
        print('Found an evaluation checkpoint, but it was computed against a DIFFERENT '
              'trained model or test set -- discarding it rather than risk mixing results '
              'from two different models.')
        evasion_results = {cls_idx: {'total': 0, 'evaded': 0, 'steps_to_evade': []} for cls_idx in range(1, 6)}
        X_adv_ppo_full = np.zeros_like(X_attack_test)
        start_i = 0
else:
    evasion_results = {cls_idx: {'total': 0, 'evaded': 0, 'steps_to_evade': []} for cls_idx in range(1, 6)}
    X_adv_ppo_full = np.zeros_like(X_attack_test)
    start_i = 0
    print(f'Starting fresh: 0/{N_EVAL_EPISODES} samples.')

import time as _time
_t0 = _time.time()

for i in range(start_i, N_EVAL_EPISODES):
    eval_env.X_pool = X_attack_test[i:i+1]
    eval_env.y_pool = y_attack_test[i:i+1]
    eval_env.rng = np.random.RandomState(0)

    obs, _ = eval_env.reset()
    lstm_states = None
    episode_starts = np.ones((1,), dtype=bool)

    true_class = int(y_attack_test[i])
    evasion_results[true_class]['total'] += 1

    final_obs = obs

    for step in range(MAX_STEPS):
        action, lstm_states = ppo_model.predict(
            obs, state=lstm_states, episode_start=episode_starts, deterministic=True
        )
        episode_starts = np.zeros((1,), dtype=bool)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        final_obs = obs

        if info.get('evaded', False):
            evasion_results[true_class]['evaded'] += 1
            evasion_results[true_class]['steps_to_evade'].append(step + 1)
            break
        if terminated or truncated:
            break

    X_adv_ppo_full[i] = np.asarray(final_obs).reshape(-1)[:N_FEATURES]

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == N_EVAL_EPISODES:
        joblib.dump({'evasion_results': evasion_results, 'X_adv_ppo_full': X_adv_ppo_full,
                     'next_i': i + 1, 'fingerprint': _eval_fp}, CHECKPOINT_PATH)
        elapsed = _time.time() - _t0
        rate = (i + 1 - start_i) / elapsed if elapsed > 0 else 0
        remaining = (N_EVAL_EPISODES - i - 1) / rate if rate > 0 else float('nan')
        print(f'  {i+1}/{N_EVAL_EPISODES} done ({rate:.1f} samples/s, '
              f'~{remaining/60:.1f} min remaining) -- checkpoint saved')

print(f"\n{'Class':<14} {'Total':>8} {'Evaded':>8} {'Evasion Rate':>14} {'Mean Steps':>12}")
for cls_idx in range(1, 6):
    r = evasion_results[cls_idx]
    rate = r['evaded'] / r['total'] if r['total'] > 0 else 0.0
    mean_steps = np.mean(r['steps_to_evade']) if r['steps_to_evade'] else float('nan')
    print(f"{class_names[cls_idx]:<14} {r['total']:>8} {r['evaded']:>8} {rate:>14.4f} {mean_steps:>12.2f}")

np.save(f'X_adv_ppo_full_baseline_{PIPELINE_VERSION}.npy', X_adv_ppo_full)
print(f"\nSaved X_adv_ppo_full_baseline_{PIPELINE_VERSION}.npy: shape {X_adv_ppo_full.shape}")

# Clean up the checkpoint now that the real output is saved -- keeping a stale
# checkpoint around risks silently resuming from old data on a future re-run.
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print(f'Removed checkpoint: {CHECKPOINT_PATH} (superseded by final results)')


## 7. Comparison: PPO vs. PGD evasion rates (Attack 1 vs. Attack 2)

Loads real Attack 1 PGD evasion numbers from `attack1_results_v4.pkl` (eps=0.1) instead
of hardcoding stale values.

In [ ]:
# CHANGED: cross-reference the baseline-model Attack 1 run, not the tuned-model one,
# so this is an apples-to-apples PGD-vs-PPO comparison for the same (baseline) model.
attack1_results = joblib.load(f'attack1_results_baseline_{PIPELINE_VERSION}.pkl')
pgd_evasion_by_name = attack1_results['per_class_evasion_pgd_eps01']
class_idx_to_name = {1: 'backdoor', 2: 'Bruteforce', 3: 'FDI', 4: 'ransomware', 5: 'reverseshell'}
pgd_evasion = {cls_idx: pgd_evasion_by_name[class_idx_to_name[cls_idx]] for cls_idx in range(1, 6)}

print(f"{'Class':<14} {'PGD Evasion':>12} {'PPO Evasion':>12} {'Diff':>8}")
for cls_idx in range(1, 6):
    ppo_rate = evasion_results[cls_idx]['evaded'] / evasion_results[cls_idx]['total']
    pgd_rate = pgd_evasion[cls_idx]
    diff = ppo_rate - pgd_rate
    print(f"{class_names[cls_idx]:<14} {pgd_rate:>12.4f} {ppo_rate:>12.4f} {diff:>+8.4f}")

print('\nInterpretation:')
print('  - If PPO unlocks evasion for FDI/ransomware/reverseshell (PGD near 0.0),')
print('    this demonstrates the value of iterative, black-box, adaptive attacks')
print('    beyond single-shot gradient-based perturbation, and would weaken the')
print('    physics-immunity finding from Attack 1 (immunity would be PGD-specific,')
print('    not a genuine architectural property).')
print('  - If PPO concentrates on/improves backdoor/Bruteforce (already PGD-evadable)')
print('    while FDI/ransomware/reverseshell remain resistant, this strengthens the')
print('    physics-immunity finding -- the resistance holds across attack methodologies,')
print('    not just against gradient-based perturbation.')

## 7b. Budget-matched PPO: does physics-immunity survive a fair perturbation budget?

Section 7's comparison is confounded: PGD's perturbation is clamped to a total L_inf
budget of eps=0.1 around the original input (`pgd_attack` in NB3 re-clamps at every
iteration), while PPO's environment only clips each individual step to +/-0.3 and lets
displacement accumulate freely across up to 10 steps -- a per-episode budget up to
~30x larger than PGD's, unchanged from the leakage-era reference architecture. Some
(not necessarily all) of Section 7's evasion gap could simply be a larger perturbation
allowance rather than a smarter black-box attack methodology.

This section retrains and re-evaluates PPO with the SAME total budget as PGD
(`eps_total_budget=0.1`, via the `IDSEvasionEnv` extension added in Section 3), on the
FULL attack-class test set -- so this is a genuine apples-to-apples comparison against
Attack 1's PGD numbers, not a subsample. The agent still gets up to 10 steps to search
for the best perturbation *within* that budget (matching PGD's own 10 gradient
iterations in degrees of freedom), it just cannot exceed the budget in total.

Both the original (unconstrained) and this budget-matched PPO result are kept and
reported side by side -- the unconstrained number shows PPO's ceiling if given a free
hand (matches the original reference architecture), the budget-matched number isolates
attack-methodology sophistication from budget size, and is the number the
physics-immunity claim actually needs to survive.

In [ ]:
# ADDED: same mid-training checkpoint/resume pattern as cell 5.
CKPT_DIR_BUD = f'_ppo_train_checkpoints_budgeted_{PIPELINE_VERSION}'
os.makedirs(CKPT_DIR_BUD, exist_ok=True)

EPS_TOTAL_BUDGET = 0.1  # matches Attack 1's PGD eps=0.1 exactly

env_budgeted = IDSEvasionEnv(
    X_pool=X_attack_train, y_pool=y_attack_train,
    model=xgb_model, cyber_idx=CYBER_IDX, n_features=N_FEATURES, n_classes=6,
    seed=RANDOM_STATE, eps_total_budget=EPS_TOTAL_BUDGET,
)

_current_fp_bud = _fingerprint_config(
    _model_fingerprint(xgb_model, X_attack_train[:200]),
    EPS_BOUND, MAX_STEPS, EVASION_REWARD, STEP_PENALTY_SCALE, TIMEOUT_PENALTY,
    RANDOM_STATE, EPS_TOTAL_BUDGET,
)
_fp_path_bud = os.path.join(CKPT_DIR_BUD, '_fingerprint.txt')

_found_bud = _latest_checkpoint(CKPT_DIR_BUD)
if (_found_bud is not None and os.path.exists(_fp_path_bud)
        and open(_fp_path_bud).read().strip() == _current_fp_bud):
    _ckpt_path_bud, _ckpt_steps_bud = _found_bud
    print(f'Found mid-training checkpoint at {_ckpt_steps_bud}/{TOTAL_TIMESTEPS} timesteps '
          f'(fingerprint matches current config) -- resuming.')
    ppo_model_budgeted = RecurrentPPO.load(_ckpt_path_bud, env=env_budgeted)
    _remaining_bud = TOTAL_TIMESTEPS - ppo_model_budgeted.num_timesteps
else:
    if _found_bud is not None:
        print('Found a checkpoint, but its fingerprint does NOT match the current model/config '
              '-- discarding it rather than risk resuming inconsistent progress.')
        import shutil as _shutil_stale_bud
        _shutil_stale_bud.rmtree(CKPT_DIR_BUD)
        os.makedirs(CKPT_DIR_BUD, exist_ok=True)
    print(f'No checkpoint found -- starting fresh, 0/{TOTAL_TIMESTEPS} timesteps.')
    with open(_fp_path_bud, 'w') as _f:
        _f.write(_current_fp_bud)
    ppo_model_budgeted = RecurrentPPO(
        'MlpLstmPolicy',
        env_budgeted,
        n_steps=128,
        batch_size=64,
        learning_rate=3e-4,
        gamma=0.99,
        seed=RANDOM_STATE,
        verbose=0,
        policy_kwargs=dict(lstm_hidden_size=64, n_lstm_layers=1),
    )
    _remaining_bud = TOTAL_TIMESTEPS

if _remaining_bud > 0:
    ckpt_callback_bud = CheckpointCallback(
        save_freq=CKPT_FREQ, save_path=CKPT_DIR_BUD, name_prefix='ppo_budgeted',
    )
    ppo_model_budgeted.learn(total_timesteps=_remaining_bud, reset_num_timesteps=False,
                              callback=ckpt_callback_bud, progress_bar=True)
else:
    print('Training already complete per checkpoint -- skipping learn().')

ppo_model_budgeted.save('ppo_evasion_agent_budgeted_baseline_v4')
print("Saved: ppo_evasion_agent_budgeted_baseline_v4.zip")

_check = RecurrentPPO.load('ppo_evasion_agent_budgeted_baseline_v4')
print('Reload check passed -- model file is valid and loadable.')
del _check

if os.path.exists(CKPT_DIR_BUD):
    _shutil.rmtree(CKPT_DIR_BUD)
    print(f'Removed training checkpoints: {CKPT_DIR_BUD} (superseded by final saved model)')

print('Budget-matched PPO training complete.')


In [ ]:
# ADDED: same checkpointing + resume pattern as cell 6.
CHECKPOINT_EVERY = 2000
CHECKPOINT_PATH_BUD = f'_checkpoint_budgeted_eval_{PIPELINE_VERSION}.pkl'

eval_env_budgeted = IDSEvasionEnv(
    X_pool=X_attack_test, y_pool=y_attack_test,
    model=xgb_model, cyber_idx=CYBER_IDX, n_features=N_FEATURES, n_classes=6,
    seed=RANDOM_STATE + 1, eps_total_budget=EPS_TOTAL_BUDGET,
)

# ADDED: same fingerprint guard as cell 7.
_eval_fp_bud = _fingerprint_config(
    _fingerprint_file('ppo_evasion_agent_budgeted_baseline_v4.zip'),
    X_attack_test.shape, float(X_attack_test[:10].sum()),
)

if os.path.exists(CHECKPOINT_PATH_BUD):
    _ckpt = joblib.load(CHECKPOINT_PATH_BUD)
    if _ckpt.get('fingerprint') == _eval_fp_bud:
        evasion_results_budgeted = _ckpt['evasion_results_budgeted']
        start_i_bud = _ckpt['next_i']
        print(f'Resuming from checkpoint: {start_i_bud}/{len(X_attack_test)} samples already done '
              f'(fingerprint matches current trained model).')
    else:
        print('Found an evaluation checkpoint, but it was computed against a DIFFERENT '
              'trained model or test set -- discarding it.')
        evasion_results_budgeted = {cls_idx: {'total': 0, 'evaded': 0} for cls_idx in range(1, 6)}
        start_i_bud = 0
else:
    evasion_results_budgeted = {cls_idx: {'total': 0, 'evaded': 0} for cls_idx in range(1, 6)}
    start_i_bud = 0
    print(f'Starting fresh: 0/{len(X_attack_test)} samples.')

import time as _time
_t0 = _time.time()

for i in range(start_i_bud, len(X_attack_test)):
    eval_env_budgeted.X_pool = X_attack_test[i:i+1]
    eval_env_budgeted.y_pool = y_attack_test[i:i+1]
    eval_env_budgeted.rng = np.random.RandomState(0)

    obs, _ = eval_env_budgeted.reset()
    lstm_states = None
    episode_starts = np.ones((1,), dtype=bool)

    true_class = int(y_attack_test[i])
    evasion_results_budgeted[true_class]['total'] += 1

    for step in range(MAX_STEPS):
        action, lstm_states = ppo_model_budgeted.predict(
            obs, state=lstm_states, episode_start=episode_starts, deterministic=True
        )
        episode_starts = np.zeros((1,), dtype=bool)
        obs, reward, terminated, truncated, info = eval_env_budgeted.step(action)

        if info.get('evaded', False):
            evasion_results_budgeted[true_class]['evaded'] += 1
            break
        if terminated or truncated:
            break

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(X_attack_test):
        joblib.dump({'evasion_results_budgeted': evasion_results_budgeted, 'next_i': i + 1,
                     'fingerprint': _eval_fp_bud}, CHECKPOINT_PATH_BUD)
        elapsed = _time.time() - _t0
        rate = (i + 1 - start_i_bud) / elapsed if elapsed > 0 else 0
        remaining = (len(X_attack_test) - i - 1) / rate if rate > 0 else float('nan')
        print(f'  {i+1}/{len(X_attack_test)} done ({rate:.1f} samples/s, '
              f'~{remaining/60:.1f} min remaining) -- checkpoint saved')

print(f"\n{'Class':<14} {'PGD (eps=0.1)':>14} {'PPO unconstrained':>18} {'PPO budget-matched':>19}")
ppo_budgeted_evasion = {}
for cls_idx in range(1, 6):
    pgd_rate = pgd_evasion[cls_idx]
    ppo_unconstrained_rate = evasion_results[cls_idx]['evaded'] / evasion_results[cls_idx]['total']
    r = evasion_results_budgeted[cls_idx]
    ppo_budgeted_rate = r['evaded'] / r['total'] if r['total'] > 0 else 0.0
    ppo_budgeted_evasion[cls_idx] = ppo_budgeted_rate
    print(f"{class_names[cls_idx]:<14} {pgd_rate:>14.4f} {ppo_unconstrained_rate:>18.4f} {ppo_budgeted_rate:>19.4f}")

print('\nInterpretation:')
print('  - If PPO budget-matched evasion collapses back toward PGD levels (especially')
print('    for FDI/ransomware/reverseshell), Section 7\'s conclusion was substantially')
print('    a budget artifact -- physics-immunity may still hold at EQUAL perturbation')
print('    budgets, and the earlier "black-box RL breaks physics-immunity" framing')
print('    needs walking back to "a LARGER perturbation budget breaks physics-immunity,')
print('    regardless of attack methodology."')
print('  - If PPO budget-matched evasion remains far above PGD (still near-total for')
print('    ransomware/reverseshell), this confirms black-box iterative search is a')
print('    genuinely more effective methodology even at an equal budget, and Section 7\'s')
print('    original conclusion holds.')

budget_matched_comparison = {
    class_names[cls_idx]: {
        'pgd_evasion': pgd_evasion[cls_idx],
        'ppo_unconstrained_evasion': evasion_results[cls_idx]['evaded'] / evasion_results[cls_idx]['total'],
        'ppo_budget_matched_evasion': ppo_budgeted_evasion[cls_idx],
    } for cls_idx in range(1, 6)
}

if os.path.exists(CHECKPOINT_PATH_BUD):
    os.remove(CHECKPOINT_PATH_BUD)
    print(f'\nRemoved checkpoint: {CHECKPOINT_PATH_BUD} (superseded by final results)')


In [ ]:
# Save the budget-matched PPO agent -- needed by Defence 1, which loads it as a
# separate artifact (notebooks don't share session state).
ppo_model_budgeted.save('ppo_evasion_agent_budgeted_baseline_v4')
print("Saved: ppo_evasion_agent_budgeted_baseline_v4.zip")

In [ ]:
import os
print(os.path.exists('ppo_evasion_agent_budgeted_baseline_v4.zip'))  # should be True now

## 8. Seed-robustness check on PPO training

Section 7's 99.5%+ evasion rate on ransomware/reverseshell is a single training run
(seed=42). Given this project's precedent (Attack 1's FGSM-vs-PGD reversal, and the
Optuna tuned-vs-baseline saga before that), a single RL training run should not be
treated as a fixed number without checking it's stable across seeds.

**Compute trade-off, stated explicitly**: retraining the full 50,000-timestep PPO
agent 5x is unavoidable (that's what actually tests training-run variance), but
re-running the FULL attack-class test-set evaluation (Section 6's ~419,101-sample
sweep) 5x would make this by far the most expensive notebook in the pipeline. Instead,
each retrained agent is evaluated on a **fixed, stratified subsample** (300 samples per
attack class = 1,500 total), built once with its own dedicated seed so every retrained
agent is judged against the *same* held-out samples -- a fair comparison across seeds,
just not a full-test-set re-evaluation. Section 6's original per-class numbers remain
the full-test-set reference; this section checks whether they're representative.

In [ ]:
# ADDED: per-seed checkpointing. This cell previously trained and evaluated 5 full
# PPO agents back to back in memory with no saved state between seeds -- a
# disconnect after seed 4 of 5 meant redoing all 5 from scratch. Now each seed's
# result is saved to disk as soon as it completes, and the loop skips any seed
# whose result is already on disk, so a resumed run picks up exactly where it left
# off. This is the single most expensive cell in the notebook (5x the training cost
# of cells 5/8 combined) and the most likely place for a long session to disconnect.

SEEDS_ROBUSTNESS = [42, 7, 123, 2024, 99]
N_EVAL_PER_CLASS_SUBSAMPLE = 300
SEED_CHECKPOINT_DIR = f'_checkpoint_seed_robustness_{PIPELINE_VERSION}'
os.makedirs(SEED_CHECKPOINT_DIR, exist_ok=True)

subsample_rng = np.random.RandomState(2025)
subsample_idx_per_class = {}
for cls_idx in range(1, 6):
    cls_mask = (y_attack_test == cls_idx)
    cls_positions = np.where(cls_mask)[0]
    n_take = min(N_EVAL_PER_CLASS_SUBSAMPLE, len(cls_positions))
    subsample_idx_per_class[cls_idx] = subsample_rng.choice(cls_positions, size=n_take, replace=False)

all_subsample_idx = np.concatenate([subsample_idx_per_class[c] for c in range(1, 6)])
X_eval_subsample = X_attack_test[all_subsample_idx]
y_eval_subsample = y_attack_test[all_subsample_idx]
print(f'Fixed evaluation subsample: {len(X_eval_subsample)} samples '
      f'({N_EVAL_PER_CLASS_SUBSAMPLE} per class, capped by class size)')

seed_results = []
# ADDED: fingerprint covering the model + the fixed evaluation subsample, so a
# leftover per-seed result from a different model/config is never silently reused.
_seed_level_fp = _fingerprint_config(
    _model_fingerprint(xgb_model, X_attack_train[:200]),
    EPS_BOUND, MAX_STEPS, EVASION_REWARD, STEP_PENALTY_SCALE, TIMEOUT_PENALTY,
    X_eval_subsample.shape, float(X_eval_subsample[:10].sum()),
)

for seed in SEEDS_ROBUSTNESS:
    _seed_ckpt_path = os.path.join(SEED_CHECKPOINT_DIR, f'seed_{seed}.pkl')

    if os.path.exists(_seed_ckpt_path):
        seed_result = joblib.load(_seed_ckpt_path)
        if seed_result.get('fingerprint') == _seed_level_fp:
            seed_results.append(seed_result)
            seed_evasion = seed_result['per_class_evasion']
            print(f'  seed={seed:<5}  [already done, loaded from checkpoint]  ' + '  '.join(
                f'{class_names[c]}={seed_evasion[c]:.3f}' for c in range(1, 6)))
            continue
        else:
            print(f'  seed={seed:<5}  [stale checkpoint found -- model/config changed, redoing]')

    # --- retrain PPO at this seed, with mid-training checkpoint/resume ---
    env_seed = IDSEvasionEnv(
        X_pool=X_attack_train, y_pool=y_attack_train,
        model=xgb_model, cyber_idx=CYBER_IDX, n_features=N_FEATURES, n_classes=6,
        seed=seed,
    )
    _ckpt_dir_seed = os.path.join(SEED_CHECKPOINT_DIR, f'train_ckpt_seed_{seed}')
    os.makedirs(_ckpt_dir_seed, exist_ok=True)
    _fp_path_seed = os.path.join(_ckpt_dir_seed, '_fingerprint.txt')
    _current_fp_seed = _fingerprint_config(_seed_level_fp, seed)
    _found_seed = _latest_checkpoint(_ckpt_dir_seed)
    if (_found_seed is not None and os.path.exists(_fp_path_seed)
            and open(_fp_path_seed).read().strip() == _current_fp_seed):
        _ckpt_path_seed, _ckpt_steps_seed = _found_seed
        print(f'    [seed {seed}] resuming training from {_ckpt_steps_seed}/{TOTAL_TIMESTEPS} timesteps '
              f'(fingerprint matches)')
        ppo_model_seed = RecurrentPPO.load(_ckpt_path_seed, env=env_seed)
        _remaining_seed = TOTAL_TIMESTEPS - ppo_model_seed.num_timesteps
    else:
        if _found_seed is not None:
            print(f'    [seed {seed}] stale training checkpoint found -- discarding')
            import shutil as _shutil_stale_seed
            _shutil_stale_seed.rmtree(_ckpt_dir_seed)
            os.makedirs(_ckpt_dir_seed, exist_ok=True)
        with open(_fp_path_seed, 'w') as _f:
            _f.write(_current_fp_seed)
        ppo_model_seed = RecurrentPPO(
            'MlpLstmPolicy', env_seed,
            n_steps=128, batch_size=64, learning_rate=3e-4, gamma=0.99,
            seed=seed, verbose=0,
            policy_kwargs=dict(lstm_hidden_size=64, n_lstm_layers=1),
        )
        _remaining_seed = TOTAL_TIMESTEPS
    if _remaining_seed > 0:
        _ckpt_cb_seed = CheckpointCallback(save_freq=CKPT_FREQ, save_path=_ckpt_dir_seed,
                                            name_prefix=f'ppo_seed{seed}')
        ppo_model_seed.learn(total_timesteps=_remaining_seed, reset_num_timesteps=False,
                              callback=_ckpt_cb_seed, progress_bar=False)
    import shutil as _shutil2
    if os.path.exists(_ckpt_dir_seed):
        _shutil2.rmtree(_ckpt_dir_seed)

    # --- evaluate on the FIXED subsample (same samples for every seed) ---
    eval_env_seed = IDSEvasionEnv(
        X_pool=X_eval_subsample, y_pool=y_eval_subsample,
        model=xgb_model, cyber_idx=CYBER_IDX, n_features=N_FEATURES, n_classes=6,
        seed=seed + 1000,
    )
    per_class_seed = {cls_idx: {'total': 0, 'evaded': 0} for cls_idx in range(1, 6)}

    for i in range(len(X_eval_subsample)):
        eval_env_seed.X_pool = X_eval_subsample[i:i+1]
        eval_env_seed.y_pool = y_eval_subsample[i:i+1]
        eval_env_seed.rng = np.random.RandomState(0)

        obs, _ = eval_env_seed.reset()
        lstm_states = None
        episode_starts = np.ones((1,), dtype=bool)
        true_class = int(y_eval_subsample[i])
        per_class_seed[true_class]['total'] += 1

        for step in range(MAX_STEPS):
            action, lstm_states = ppo_model_seed.predict(
                obs, state=lstm_states, episode_start=episode_starts, deterministic=True
            )
            episode_starts = np.zeros((1,), dtype=bool)
            obs, reward, terminated, truncated, info = eval_env_seed.step(action)
            if info.get('evaded', False):
                per_class_seed[true_class]['evaded'] += 1
                break
            if terminated or truncated:
                break

    seed_evasion = {cls_idx: per_class_seed[cls_idx]['evaded'] / per_class_seed[cls_idx]['total']
                     for cls_idx in range(1, 6)}
    seed_result = {'seed': seed, 'per_class_evasion': seed_evasion}

    # Save this seed's result to disk immediately, before moving to the next seed.
    seed_result['fingerprint'] = _seed_level_fp
    joblib.dump(seed_result, _seed_ckpt_path)
    seed_results.append(seed_result)

    print(f"  seed={seed:<5}  " + "  ".join(
        f"{class_names[c]}={seed_evasion[c]:.3f}" for c in range(1, 6)) + '  [checkpoint saved]')

print(f'\nPer-class evasion rate across {len(SEEDS_ROBUSTNESS)} seeds '
      f'(subsample n={len(X_eval_subsample)}, {N_EVAL_PER_CLASS_SUBSAMPLE}/class):')
print(f"{'Class':<14} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Range':>8}")
seed_robustness_multiclass = {'seeds': SEEDS_ROBUSTNESS, 'per_seed_results': seed_results,
                               'n_eval_per_class': N_EVAL_PER_CLASS_SUBSAMPLE, 'per_class_stats': {}}
for cls_idx in range(1, 6):
    vals = np.array([r['per_class_evasion'][cls_idx] for r in seed_results])
    stats = {'mean': float(vals.mean()), 'std': float(vals.std()),
              'min': float(vals.min()), 'max': float(vals.max()), 'range': float(vals.max() - vals.min())}
    seed_robustness_multiclass['per_class_stats'][class_names[cls_idx]] = stats
    print(f"{class_names[cls_idx]:<14} {stats['mean']:>8.4f} {stats['std']:>8.4f} "
          f"{stats['min']:>8.4f} {stats['max']:>8.4f} {stats['range']:>8.4f}")

unstable_classes = [name for name, s in seed_robustness_multiclass['per_class_stats'].items() if s['std'] > 0.05]
if unstable_classes:
    print(f'\nNOTE: std > 0.05 for {unstable_classes} -- Section 6/7\'s single-seed (42) '
          f'numbers for these classes should be reported with this spread, not as fixed values.')
else:
    print(f'\nAll classes stable across seeds (std <= 0.05) -- Section 6/7\'s single-seed '
          f'results are reasonably representative for every class, including ransomware/reverseshell.')

# Clean up per-seed checkpoints now that the aggregate result is computed and will be
# saved in the final results dict by the save cell below.
import shutil as _shutil
if os.path.exists(SEED_CHECKPOINT_DIR):
    _shutil.rmtree(SEED_CHECKPOINT_DIR)
    print(f'\nRemoved checkpoint directory: {SEED_CHECKPOINT_DIR} (superseded by final results)')


## Notes

The v3/pre-leakage-fix run's reported numbers are from the leakage-inflated iteration
and are not comparable to this run's numbers. Section 7's interpretation is the key
question this attack answers for the dissertation: does physics-immunity (observed in
Attack 1) hold up against a fundamentally different, black-box, iterative attack
methodology, or was it an artifact specific to gradient-based perturbation?

## Save Attack 2 results to disk

## Comparison: baseline (untuned) vs Optuna-tuned XGBoost -- identical PPO attack methodology


In [ ]:
# Direct comparison against the original NB4 run (Optuna-tuned XGBoost, F1=0.7110),
# if attack2_results_v4.pkl from that run is present in the same directory.
import os as _os

_tuned_path = f'attack2_results_{PIPELINE_VERSION}.pkl'
if _os.path.exists(_tuned_path):
    tuned = joblib.load(_tuned_path)
    print('=' * 88)
    print('BASELINE (untuned) vs TUNED XGBoost -- identical PPO attack methodology')
    print('=' * 88)
    print(f"{'':14} {'PGD':>8} {'PPO-unc':>10} {'PPO-bud':>10}   "
          f"{'PGD':>8} {'PPO-unc':>10} {'PPO-bud':>10}")
    print(f"{'':14} {'--- baseline ---':^30}   {'--- tuned ---':^30}")
    tuned_bmc = tuned.get('budget_matched_comparison', {})
    for cls_idx in range(1, 6):
        name = class_names[cls_idx]
        b = budget_matched_comparison.get(name, {})
        t = tuned_bmc.get(name, {})
        print(f"{name:<14} "
              f"{b.get('pgd_evasion', float('nan')):>8.2%} "
              f"{b.get('ppo_unconstrained_evasion', float('nan')):>10.2%} "
              f"{b.get('ppo_budget_matched_evasion', float('nan')):>10.2%}   "
              f"{t.get('pgd_evasion', float('nan')):>8.2%} "
              f"{t.get('ppo_unconstrained_evasion', float('nan')):>10.2%} "
              f"{t.get('ppo_budget_matched_evasion', float('nan')):>10.2%}")
    print()
    print('Per-class diff (baseline minus tuned), percentage points:')
    print(f"{'':14} {'PGD':>8} {'PPO-unc':>10} {'PPO-bud':>10}")
    for cls_idx in range(1, 6):
        name = class_names[cls_idx]
        b = budget_matched_comparison.get(name, {})
        t = tuned_bmc.get(name, {})
        d_pgd = (b.get('pgd_evasion', 0) - t.get('pgd_evasion', 0)) * 100
        d_unc = (b.get('ppo_unconstrained_evasion', 0) - t.get('ppo_unconstrained_evasion', 0)) * 100
        d_bud = (b.get('ppo_budget_matched_evasion', 0) - t.get('ppo_budget_matched_evasion', 0)) * 100
        print(f"{name:<14} {d_pgd:>+7.2f}pp {d_unc:>+9.2f}pp {d_bud:>+9.2f}pp")
    print()
    print('Interpretation guide: the central physics-immunity finding is ransomware/reverseshell')
    print('at 0.00% under both PGD and budget-matched PPO. Check whether that holds for BOTH')
    print('models above. Large diffs concentrated in specific classes (as seen for backdoor in')
    print('the Attack 1 baseline check) would mean the tuned/baseline choice materially affects')
    print('the PPO findings too, not just PGD -- worth flagging explicitly if so, following the')
    print('same pattern already applied to Attack 1 in the dissertation.')
else:
    print(f'{_tuned_path} not found in this directory -- run the original NB4 '
          f'(Optuna-tuned) alongside this one to generate the comparison.')


In [ ]:
import os

ppo_model.save(f'ppo_evasion_agent_baseline_{PIPELINE_VERSION}')

attack2_results = {
    cls_idx: {
        'evasion_rate': evasion_results[cls_idx]['evaded'] / evasion_results[cls_idx]['total'],
        'total':        evasion_results[cls_idx]['total'],
        'evaded':       evasion_results[cls_idx]['evaded'],
        'mean_steps':   float(np.mean(evasion_results[cls_idx]['steps_to_evade']))
                        if evasion_results[cls_idx]['steps_to_evade'] else 0.0,
    }
    for cls_idx in range(1, 6)
}
attack2_results['PIPELINE_VERSION'] = PIPELINE_VERSION
attack2_results['pgd_comparison'] = {class_names[cls_idx]: {
    'pgd_evasion': pgd_evasion[cls_idx],
    'ppo_evasion': evasion_results[cls_idx]['evaded'] / evasion_results[cls_idx]['total'],
} for cls_idx in range(1, 6)}
attack2_results['seed_robustness'] = seed_robustness_multiclass
attack2_results['budget_matched_comparison'] = budget_matched_comparison

joblib.dump(attack2_results, f'attack2_results_baseline_{PIPELINE_VERSION}.pkl')

print(f"NB4-baseline ({PIPELINE_VERSION}, multiclass, UNTUNED XGBoost) artifacts saved:")
for f in [f'ppo_evasion_agent_baseline_{PIPELINE_VERSION}.zip', f'attack2_results_baseline_{PIPELINE_VERSION}.pkl',
          f'X_adv_ppo_full_baseline_{PIPELINE_VERSION}.npy']:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1e6
        print(f'  {f}  ({size:.1f} MB)')

print(f'\nAttack 2 summary:')
for cls_idx in range(1, 6):
    rate = evasion_results[cls_idx]['evaded'] / evasion_results[cls_idx]['total']
    print(f'  {class_names[cls_idx]:<14} evasion rate: {rate:.4f}')